# 演習：正規化フローによる変分推論

距離センサーの観測から物体の位置を推定する。事後分布を対角ガウス・全共分散ガウス・カップリングフローで近似して比べる。

`TODO` の行を埋める。解答を埋めたものは `exercise_solution.ipynb`。

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import trange

torch.set_default_dtype(torch.float64)

KINDS = ("diag", "full", "flow")
COLORS = {"diag": "#E07B39", "full": "#2F6DB5", "flow": "#1E9E74"}
C_TRUE = "#4A4A4A"

## アフィンカップリング層

入力 $\boldsymbol u=(u_1,u_2)$ のうち $u_1$ はそのまま通し、$u_2$ を変換する。$s,t$ は $u_1$ を入力とする小さな NN の出力。
$$z_1=u_1,\qquad z_2=u_2\,e^{s(u_1)}+t(u_1),\qquad \log|\det J|=s(u_1)$$

**TODO 1**：`z2` と `log_det` を書く。

In [ ]:
def std_normal_logprob(u):
    # log N(u; 0, I)。u: (S, D) -> (S,)
    return -0.5 * (u**2).sum(-1) - 0.5 * u.shape[-1] * math.log(2 * math.pi)


def affine_forward(u2, s, t):
    # u2, s, t: (S,) -> z2, log_det: (S,)
    z2 = ...  # TODO 1
    log_det = ...  # TODO 1
    return z2, log_det


class AffineCoupling(nn.Module):
    def __init__(self, flip: bool, hidden: int = 64):
        super().__init__()
        self.flip = flip  # True なら u2 を固定して u1 を変換する
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )
        nn.init.zeros_(self.net[-1].weight)  # 恒等写像から始める
        nn.init.zeros_(self.net[-1].bias)

    def _split(self, x):
        return (x[:, 1], x[:, 0]) if self.flip else (x[:, 0], x[:, 1])

    def _join(self, a, b):
        return torch.stack((b, a) if self.flip else (a, b), -1)

    def _st(self, a):
        s, t = self.net(a[:, None]).unbind(-1)
        return torch.tanh(s), t  # s の大きさを抑えて学習を安定させる

    def forward(self, u):
        a, b = self._split(u)
        s, t = self._st(a)
        b, log_det = affine_forward(b, s, t)
        return self._join(a, b), log_det

    def inverse(self, z):
        a, b = self._split(z)
        s, t = self._st(a)
        return self._join(a, (b - t) * torch.exp(-s)), -s

## フロー

層を順に通し、各層の $\log|\det J|$ を足す。基底分布は $q_0=\mathcal N(\boldsymbol 0,I)$。
$$\log|\det J_T|=\sum_k\log|\det J_{T_k}|$$

**TODO 2**：`log_det` に各層の `ld` を足す。

比較に使う対角ガウス $\boldsymbol z=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol u$ と全共分散ガウス $\boldsymbol z=\boldsymbol\mu+L\boldsymbol u$ も、順写像・逆写像・$\log|\det J|$ を持つ層として書く。カップリングフローは、カップリング層8枚の後に対角ガウスと同じ層を置く。どのモデルも学習前は $\mathcal N(\boldsymbol\mu,\sigma^2I)$ で、`make_model` の `mu` と `sigma` で決める。

In [ ]:
class Flow(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, u):
        # u -> z。サンプルを作る向き
        log_det = torch.zeros(u.shape[0])
        for layer in self.layers:
            u, ld = layer(u)
            log_det = ...  # TODO 2
        return u, log_det

    def inverse(self, z):
        # z -> u。任意の点の密度を測る向き
        log_det = torch.zeros(z.shape[0])
        for layer in reversed(self.layers):
            z, ld = layer.inverse(z)
            log_det = log_det + ld
        return z, log_det


class DiagAffine(nn.Module):
    # z = mu + sigma * u
    def __init__(self, mu, sigma):
        super().__init__()
        self.mu = nn.Parameter(torch.tensor(mu))
        self.log_sigma = nn.Parameter(torch.full((2,), math.log(sigma)))

    def forward(self, u):
        return self.mu + torch.exp(self.log_sigma) * u, self.log_sigma.sum().expand(u.shape[0])

    def inverse(self, z):
        return (z - self.mu) * torch.exp(-self.log_sigma), -self.log_sigma.sum().expand(z.shape[0])


class FullAffine(nn.Module):
    # z = mu + L u（L は対角成分が正の下三角行列）
    def __init__(self, mu, sigma):
        super().__init__()
        self.mu = nn.Parameter(torch.tensor(mu))
        self.log_diag = nn.Parameter(torch.full((2,), math.log(sigma)))
        self.off = nn.Parameter(torch.zeros(2, 2))

    def L(self):
        return torch.tril(self.off, -1) + torch.diag(torch.exp(self.log_diag))

    def forward(self, u):
        return self.mu + u @ self.L().T, self.log_diag.sum().expand(u.shape[0])

    def inverse(self, z):
        u = torch.linalg.solve_triangular(self.L(), (z - self.mu).T, upper=False).T
        return u, -self.log_diag.sum().expand(z.shape[0])


def make_model(kind, mu=(2.0, 2.0), sigma=1.0, n_layers=8):
    match kind:
        case "diag":
            layers = [DiagAffine(mu, sigma)]
        case "full":
            layers = [FullAffine(mu, sigma)]
        case "flow":
            layers = [AffineCoupling(flip=k % 2 == 1) for k in range(n_layers)]
            layers.append(DiagAffine(mu, sigma))
    return Flow(layers)

可逆性と、`log_det` が自動微分で求めたヤコビアンの行列式と一致することを確かめる。

In [ ]:
def check_flow():
    torch.manual_seed(1)
    flow = make_model("flow", n_layers=4)
    for p in flow.parameters():  # 恒等写像にならないようにパラメータを乱す
        p.data += 0.3 * torch.randn_like(p)
    u = torch.randn(5, 2)
    z, log_det = flow(u)
    u_back, log_det_inv = flow.inverse(z)
    assert torch.allclose(u, u_back), "逆写像で元に戻らない"
    for i in range(5):
        J = torch.autograd.functional.jacobian(lambda x: flow(x[None])[0][0], u[i])
        assert torch.allclose(log_det[i], torch.linalg.slogdet(J)[1]), "log_det がヤコビアンと一致しない"
    assert torch.allclose(log_det, -log_det_inv)
    print("OK")


check_flow()

## 位置推定のモデル

センサー $\boldsymbol a_k$ が物体の位置 $\boldsymbol z\in\mathbb R^2$ までの距離をノイズ付きで測る。
$$\boldsymbol z\sim\mathcal N(\boldsymbol m_0,\,s_0^2I),\qquad y_k=\lVert\boldsymbol z-\boldsymbol a_k\rVert+\varepsilon_k,\quad \varepsilon_k\sim\mathcal N(0,\sigma_{\mathrm{obs}}^2)$$

まずセンサー1つ：$\boldsymbol a_1=(0,0)$、$y_1=2$、$\boldsymbol m_0=(2,2)$、$s_0=1$、$\sigma_{\mathrm{obs}}=0.1$。距離しかわからないので、事後分布はセンサーを中心とする円弧になる。

2次元なので、格子で積分して $\log p(\boldsymbol y)$ と事後密度を求められる。これを使って $\mathrm{KL}=\log p(\boldsymbol y)-\mathrm{ELBO}$ を計算する。

In [ ]:
@dataclass
class Setting:
    anchors: torch.Tensor  # センサーの位置 (K, 2)
    y: torch.Tensor  # 測った距離 (K,)
    m0: torch.Tensor  # 事前分布の平均 (2,)
    s0: float  # 事前分布の標準偏差
    sigma_obs: float = 0.1
    xlim: tuple = (-1.5, 4.0)  # 図の範囲
    ylim: tuple = (-2.5, 4.0)

    def log_joint(self, z):
        # log p(y, z) = log p(z) + log p(y | z)。z: (S, 2) -> (S,)
        log_prior = std_normal_logprob((z - self.m0) / self.s0) - 2 * math.log(self.s0)
        dist = torch.linalg.norm(z[:, None, :] - self.anchors, dim=-1)  # (S, K)
        r = (self.y - dist) / self.sigma_obs
        log_lik = (-0.5 * r**2 - math.log(self.sigma_obs * math.sqrt(2 * math.pi))).sum(-1)
        return log_prior + log_lik


@dataclass
class Reference:
    Z1: torch.Tensor
    Z2: torch.Tensor
    post: torch.Tensor  # 格子上の事後密度
    log_py: float


def reference(setting, lim=(-6.0, 6.0), n=1001):
    # 格子上で p(y, z) を積分し，log p(y) と事後密度を求める
    g = torch.linspace(*lim, n)
    Z1, Z2 = torch.meshgrid(g, g, indexing="xy")
    lj = setting.log_joint(torch.stack([Z1.ravel(), Z2.ravel()], -1)).reshape(n, n)
    log_py = torch.logsumexp(lj.ravel(), 0) + 2 * torch.log(g[1] - g[0])
    return Reference(Z1, Z2, torch.exp(lj - log_py), log_py.item())


def plot_posterior(ax, setting, ref, title=""):
    ax.contour(ref.Z1, ref.Z2, ref.post, levels=6, colors=C_TRUE, linewidths=0.8)
    ax.scatter(*setting.anchors.T, marker="^", s=80, c="k", zorder=3)
    ax.set(aspect="equal", xlim=setting.xlim, ylim=setting.ylim, title=title)


one = Setting(
    anchors=torch.tensor([[0.0, 0.0]]),
    y=torch.tensor([2.0]),
    m0=torch.tensor([2.0, 2.0]),
    s0=1.0,
)
ref_one = reference(one)

fig, ax = plt.subplots(figsize=(4, 4))
plot_posterior(ax, one, ref_one, f"true posterior, log p(y) = {ref_one.log_py:.3f}")
plt.show()

## 変分推論

$$\mathrm{ELBO}=\mathbb E_{\boldsymbol u\sim q_0}\bigl[\log p(\boldsymbol y,\boldsymbol z)-\log q(\boldsymbol z)\bigr],\qquad \boldsymbol z=T(\boldsymbol u)$$

$\boldsymbol u$ を引いて写像に通すと、$\boldsymbol z$ と $\log|\det J_T(\boldsymbol u)|$ が得られる。
$$\log q(\boldsymbol z)=\log q_0(\boldsymbol u)-\log|\det J_T(\boldsymbol u)|$$

**TODO 3**：`log_q` を書く。`std_normal_logprob(u)` と `log_det` を使う。

In [ ]:
def neg_elbo(model, setting, S=256):
    u = torch.randn(S, 2)
    z, log_det = model(u)
    log_q = ...  # TODO 3
    return (log_q - setting.log_joint(z)).mean()


def fit(model, setting, steps=3000, lr=5e-3, desc=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bar = trange(steps, desc=desc)
    for step in bar:
        loss = neg_elbo(model, setting)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 100 == 0:
            bar.set_postfix(loss=f"{loss.item():.3f}")
    return model


@torch.no_grad()
def elbo(model, setting, S=200_000):
    return -neg_elbo(model, setting, S).item()


models = {}
for kind in KINDS:
    torch.manual_seed(0)
    models[kind] = fit(make_model(kind, mu=(2.0, 2.0), sigma=1.0), one, desc=kind)
    e = elbo(models[kind], one)
    print(f"{kind}: ELBO {e:.3f}, KL {ref_one.log_py - e:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, kind in zip(axes, KINDS):
    with torch.no_grad():
        z, _ = models[kind](torch.randn(2000, 2))
    plot_posterior(ax, one, ref_one, kind)
    ax.scatter(*z.T, s=2, alpha=0.3, c=COLORS[kind])
plt.tight_layout()
plt.show()

## 密度の図

散布図では密度の形を読み取りにくいので、格子点 $\boldsymbol z$ で $q(\boldsymbol z)$ を評価して描く。格子点では元の $\boldsymbol u$ がわからないので、逆写像で $\boldsymbol u=T^{-1}(\boldsymbol z)$ に戻す。
$$\log q(\boldsymbol z)=\log q_0(\boldsymbol u)+\log|\det J_{T^{-1}}(\boldsymbol z)|$$
`model.inverse(z)` は $\boldsymbol u$ と $\log|\det J_{T^{-1}}(\boldsymbol z)|$ を返す。

**TODO 4**：`log_q` を書く。TODO 3 とは写像の向きと `log_det` の符号が違う。

In [ ]:
def log_density(model, z):
    u, log_det = model.inverse(z)
    log_q = ...  # TODO 4
    return log_q


def plot_density(ax, model, setting, ref, color, title=""):
    g1 = torch.linspace(*setting.xlim, 200)
    g2 = torch.linspace(*setting.ylim, 200)
    G1, G2 = torch.meshgrid(g1, g2, indexing="xy")
    with torch.no_grad():
        q = torch.exp(log_density(model, torch.stack([G1.ravel(), G2.ravel()], -1)))
    cmap = LinearSegmentedColormap.from_list("q", ["white", color])
    ax.pcolormesh(G1, G2, q.reshape(G1.shape), shading="auto", cmap=cmap)
    plot_posterior(ax, setting, ref, title)


fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, kind in zip(axes, KINDS):
    plot_density(ax, models[kind], one, ref_one, COLORS[kind], kind)
plt.tight_layout()
plt.show()

## センサー2つの場合

センサーを $(0,0)$ と $(3,0.5)$ に置き、どちらも距離 2 を観測する。2つの円の交点は2つあり、観測からはどちらか決まらないので、事後分布は二峰になる。事前分布の中心 $(1.5,0)$ に近い下の山の方が確率が大きい（下 0.57、上 0.43）。

学習前の分布 $\mathcal N(\boldsymbol\mu_{\mathrm{init}},\sigma_{\mathrm{init}}^2I)$ を `init_mu` と `init_sigma` で変えて、学習前後の密度を比べる。

In [ ]:
two = Setting(
    anchors=torch.tensor([[0.0, 0.0], [3.0, 0.5]]),
    y=torch.tensor([2.0, 2.0]),
    m0=torch.tensor([1.5, 0.0]),
    s0=1.5,
    xlim=(-1.5, 4.5),
    ylim=(-3.0, 3.0),
)
ref_two = reference(two)

In [ ]:
init_mu = (1.5, 0.0)  # 学習前の分布の中心
init_sigma = 1.0  # 学習前の分布の幅

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, kind in enumerate(KINDS):
    torch.manual_seed(0)
    model = make_model(kind, init_mu, init_sigma)
    for row, stage in enumerate(("before", "after")):
        if stage == "after":
            fit(model, two, desc=kind)
        e = elbo(model, two)
        title = f"{kind} ({stage})\nELBO {e:.3f}, KL {ref_two.log_py - e:.3f}"
        plot_density(axes[row, col], model, two, ref_two, COLORS[kind], title)
plt.tight_layout()
plt.show()

## 確認

- ガウス近似の限界：曲がった事後分布をどこまで近似できるか
- reverse KL の限界：初期値によって片方の山にとらわれる
- ELBO の限界：山の見落としは ELBO の値だけではわからない